# Tokenzier

## HouseKeeping

In [4]:
import sys
import os
import subprocess

print("Python executable:", sys.executable)
print("Python prefix:", sys.prefix)
print("Conda environment:", os.environ.get('CONDA_DEFAULT_ENV'))
print("Conda prefix:", os.environ.get('CONDA_PREFIX'))

# Check pip location
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Pip location:", result.stdout.strip())

# Check pip version
result = subprocess.run(['pip', '--version'], capture_output=True, text=True)
print("Pip version:", result.stdout.strip())

# Add environment's bin directory to PATH
env_bin = os.path.dirname(sys.executable)
current_path = os.environ['PATH']
if env_bin not in current_path:
    os.environ['PATH'] = f"{env_bin}:{current_path}"

# Verify fix
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Fixed pip location:", result.stdout.strip())

Python executable: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/python3
Python prefix: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch
Conda environment: base
Conda prefix: /opt/conda
Pip location: /opt/conda/bin/pip
Pip version: pip 25.1.1 from /opt/conda/lib/python3.12/site-packages/pip (python 3.12)
Fixed pip location: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/pip


In [5]:
!pwd

/home/sagemaker-user/courses/llms-from-scratch/Chapter_2


# Code

In [6]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total Characters:", len(raw_text))

Total Characters: 20479


In [7]:
print(raw_text[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


### Split text

In [8]:
import re
text = "Kya haal chal bro? sab sahi hai?"

result = re.split(r'(\s)', text)

print(result)

['Kya', ' ', 'haal', ' ', 'chal', ' ', 'bro?', ' ', 'sab', ' ', 'sahi', ' ', 'hai?']


In [9]:
# spliting punctuation too

result = re.split(r'([,.?]|\s)', text)

print(result)

['Kya', ' ', 'haal', ' ', 'chal', ' ', 'bro', '?', '', ' ', 'sab', ' ', 'sahi', ' ', 'hai', '?', '']


In [10]:
# removing spaces

result = [ item for item in result if item.strip() ]

print(result)

['Kya', 'haal', 'chal', 'bro', '?', 'sab', 'sahi', 'hai', '?']


In [11]:
# test it all 

text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]

print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [12]:
# applying the tokenizer to complete text file

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(len(preprocessed))

4690


In [13]:
# I first got around 9000 token - forgot removed the spaces.
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


### Converting to Token ID

In [14]:
# counts the unique token 
all_words = sorted(set(preprocessed))

vocab_size = len(all_words)

vocab_size

1130

In [15]:
# Creating a vocabulary

vocab = { token:integer for integer,token in enumerate(all_words) }

# vocab 

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


### Implementing a simple text tokenizer

In [16]:
'''
#1 Stores the vocabulary as a class attribute for access in the encode and decode methods
#2 Creates an inverse vocabulary that maps token IDs back to the original text tokens
#3 Processes input text into token IDs
#4 Converts token IDs back into text
#5 Removes spaces before the specified punctuation 
'''

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self,text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]
        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

In [17]:
# calling the function 

tokenizer = SimpleTokenizerV1(vocab) # passing the mapping of word to token id

text =  """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)

print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [18]:
# Printing back the words from token id - this is what happens at the end of LLM output just before you see the output

print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [19]:
# calling the function on words that are not stored in the tokenizer dicitionary - common key value error

hinid_test_tokenizer = SimpleTokenizerV1(vocab) # passing the mapping of word to token id

text =  """"kya haal chal bhai"""

ids = hinid_test_tokenizer.encode(text)

print(ids)


KeyError: 'kya'

### Adding Custom Token

In [20]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}             

print(len(vocab.items()))

1132


In [21]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Simple text tokenizer that handles unknown words

In [22]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]
        preprocessed = [ item if item in self.str_to_int else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [23]:
# this is how they combine large document to tell the model that this is sepearate doc

text1 = "Hello, do you like tea?"

text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [24]:
tokenizerv2 = SimpleTokenizerV2(vocab)
print(tokenizerv2.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [25]:
print(tokenizerv2.decode(tokenizerv2.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


### Byte pair encoding

In [26]:
%%capture
!pip install tiktoken

In [27]:
# Version 
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [28]:
bpe_tokenizer = tiktoken.get_encoding("gpt2")

In [29]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = bpe_tokenizer.encode(text, allowed_special= {"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [30]:
print(bpe_tokenizer.decode(integers))

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [31]:
text = "Swapnil"

encode_swapnil = bpe_tokenizer.encode(text, allowed_special= {"<|endoftext|>"})

encode_swapnil

[10462, 499, 45991]

In [32]:
print(bpe_tokenizer.decode([45991]))

nil


### Data sampling with a sliding window

In [33]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = bpe_tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [34]:
# removing first 50 token from passage - optional 
enc_sample = enc_text[50:]

In [35]:
# Creating context size - The context size determines how many tokens are included in the input. 
context_size = 5

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print("x: ", x)
print("y:      ", y)  

x:  [290, 4920, 2241, 287, 257]
y:       [4920, 2241, 287, 257, 4489]


In [36]:
for i in range(1, context_size):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [37]:
for i in range(1, context_size):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(bpe_tokenizer.decode(context), "----->", bpe_tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


### A dataset for batched inputs and targets

In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetv1(Dataset): # Class inheretence
    def __init__(self, txt, bpe_tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = bpe_tokenizer.encode(txt) # Tokenized entire text

        for i in range(0, len(token_ids) - max_length, stride): # chuck the corpus into fixed size of max length with no of skiiping sliding token decide by stride
            input_chunks = token_ids[i:i + max_length]
            target_chunks = token_ids[i+1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunks))
            self.target_ids.append(torch.tensor(target_chunks))

    #3 Returns the total number of rows in the dataset 
    def __len__(self): # magic function - inhereted from parent class
        return len(self.input_ids)

    #4 Returns a single row from the dataset 
    def __getitem__(self, idx): # magic function - inhereted from parent class
        return self.input_ids[idx], self.target_ids[idx]
        

In [39]:
def create_dataloader_v1( txt, batch_size = 4, max_length=256, stride=128, shuffle=True, drop_last = True, num_workers = 0): #3 drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training. 
    tokenizer= tiktoken.get_encoding("gpt2") # initialize the tokenzier 
    
    dataset = GPTDatasetv1(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers) #4 num_workers = The number of CPU processes to use for preprocessing 
    
    return dataloader

In [40]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, shuffle=True)
data_iter = iter(dataloader)  #1 Converts dataloader into a Python iterator to fetch the next entry via Python’s built-in next() function 
first_batch = next(data_iter)
print(first_batch)

[tensor([[  262,  1633,   286, 24380,   329, 20728,   287,   257,  4286, 10273,
           438, 29370,   477,    11,   645,  1551,  1051,   286,  1683,  1719,
           587,   973,   355,   257,  8034,    13,   198,   198,   464,  1109,
          3181,  1363,   284,   502,   262,  4112,   957,  1483,   286,  3619,
           338,  2270,   351,   465,  1468,  1204,    13,   198,   198,     1,
          3987,   470,   345,  1683, 45553,   903,   351,  7521,   597,   517,
          1701,   314,  1965,    11,   991,  2045,   546,   329,   257, 12854,
           286,   884,  3842,    13,   198,   198,     1, 12295,   553,   339,
           531, 11589,    13,   198,   198,     1,  5574,  1660,    12, 49903,
           438,   273,  2123, 10813,  1701,   198,   198,  6653,  6563,  2951,
          6348,  5391,    11,   290,   465, 25839,   279,  3021,   257,  1310,
           739,   511, 22665,  4252, 10899,    13,   198,   198,     1, 12295,
           892,   286,   340,    11,   616, 13674, 

### Data loaders with different strides and context sizes

In [41]:
dataloader2 = create_dataloader_v1(raw_text, batch_size=1, max_length=8, stride=1, shuffle= True)
data_iter = iter(dataloader2)
first_batch = next(data_iter)
print(first_batch)

[tensor([[3812,  502,   13,  198,  198,    1,   40, 1549]]), tensor([[ 502,   13,  198,  198,    1,   40, 1549, 2138]])]


## Creating token embeddings

In [42]:
vocab_size = 6

output_dim = 3

torch.manual_seed(123)

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [43]:
print(embedding_layer(torch.tensor([0])))

tensor([[ 0.3374, -0.1778, -0.1690]], grad_fn=<EmbeddingBackward0>)


In [44]:
input_ids = torch.tensor([0, 1, 2, 3, 3, 4, 5])
input_ids

tensor([0, 1, 2, 3, 3, 4, 5])

In [45]:
print(embedding_layer(torch.tensor(input_ids)))

tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], grad_fn=<EmbeddingBackward0>)


/tmp/ipykernel_271968/3846183200.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  print(embedding_layer(torch.tensor(input_ids)))


## Encoding word positions

In [46]:
max_length = 4
dataloader2 = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=1, shuffle= True)
data_iter = iter(dataloader2)
inputs, targets = next(data_iter)
print("Input Token IDs : \n", inputs, "\n\n",  "Target Token IDs : \n", targets)
print("\nInputs shape:\n", inputs.shape)

Input Token IDs : 
 tensor([[ 1039,    11,   262,  4057],
        [  351,   683,    13,   314],
        [  198,  1870,   465,  8216],
        [  436,    81,  5286,  8812],
        [  670,   526,   198,   198],
        [  852,  5729, 11331, 18893],
        [  339,   531, 15376,    26],
        [  465, 12036,   780,   339]]) 

 Target Token IDs : 
 tensor([[   11,   262,  4057,  2655],
        [  683,    13,   314,   550],
        [ 1870,   465,  8216,  1297],
        [   81,  5286,  8812,  2114],
        [  526,   198,   198,    40],
        [ 5729, 11331, 18893,   540],
        [  531, 15376,    26,   788],
        [12036,   780,   339,   550]])

Inputs shape:
 torch.Size([8, 4])


In [47]:
vocab_size = 50257

output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [48]:
token_embeddings = token_embedding_layer(inputs)

print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [49]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)
print(pos_embeddings)

torch.Size([4, 256])
tensor([[-0.6303, -0.4848, -0.1366,  ...,  1.0345, -0.5012,  1.1045],
        [ 0.2062,  0.6078,  0.7187,  ..., -0.4628, -0.2319,  1.1980],
        [ 0.5806, -1.3846,  0.3266,  ...,  0.8579,  0.5059,  1.0243],
        [ 1.4323,  0.2217,  0.8599,  ...,  0.4827,  0.8459,  1.3038]],
       grad_fn=<EmbeddingBackward0>)


In [50]:
input_embeddings = token_embeddings + pos_embeddings

print(input_embeddings.shape)
print(input_embeddings)

torch.Size([8, 4, 256])
tensor([[[-2.1305, -2.4928, -0.9994,  ...,  1.6236, -0.8230,  1.6376],
         [ 1.3533,  1.3769,  1.2800,  ..., -0.2036, -0.1041,  1.7521],
         [ 0.3779, -0.9714,  2.0936,  ...,  0.5683, -0.1827,  1.3732],
         [ 2.8041,  0.4652, -0.8941,  ..., -0.0062,  1.0371,  1.2431]],

        [[-0.6375,  0.8590,  0.5917,  ...,  0.3350,  0.0906, -0.9478],
         [ 0.7009,  0.9328,  1.3268,  ...,  0.3582, -0.8879,  1.7415],
         [-0.3156, -1.7279,  1.4173,  ...,  0.7317, -2.5433,  1.9137],
         [ 3.0071, -0.2514,  0.1619,  ...,  0.4733,  1.0792,  1.1737]],

        [[-1.1991,  0.1428,  2.3197,  ...,  0.5994,  0.0639,  1.5709],
         [-0.6013,  2.2188,  1.5039,  ..., -1.3116, -0.8736,  0.7850],
         [ 1.4484,  0.1229,  0.6273,  ...,  0.1855,  0.5545,  0.5324],
         [ 1.9937,  0.1337,  0.6658,  ...,  0.4296,  1.7535,  1.3667]],

        ...,

        [[-0.3583, -1.3424, -0.8067,  ..., -1.4275, -0.0144,  2.0659],
         [ 0.2491,  1.0995,  1.52

# -----------------------------------------------------------------------------

## 13th May 2026

In [1]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")

file_path = "the-verdict.txt"

urllib.request.urlretrieve(url,file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x7f2013e49310>)

In [2]:
with open("the-verdict.txt", 'r', encoding="utf-8") as f:
    raw_text = f.read()

raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [3]:
print("Total number of characters:", len(raw_text))
print(raw_text[:300])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would ha


In [4]:
# Python regular expression library would help us to disect the words at right sections
import re
text = "Hello Smarty!, How are you doing?"
result = re.split(r'(\s)', text) # Splitting the sentence at spaces
print(result)

['Hello', ' ', 'Smarty!,', ' ', 'How', ' ', 'are', ' ', 'you', ' ', 'doing?']


In [5]:
# Let’s modify the regular expression splits on whitespaces (\s), commas, and periods ([,.]):
result = re.split(r'([,.!]|\s)', text)
print(result)

['Hello', ' ', 'Smarty', '!', '', ',', '', ' ', 'How', ' ', 'are', ' ', 'you', ' ', 'doing?']


In [6]:
result = [ item for item in result if item.strip()]
print(result)

['Hello', 'Smarty', '!', ',', 'How', 'are', 'you', 'doing?']


### When you should keep whitespaces keeping whitespaces 
``` It can be useful if we train models that are sensitive to the exact structure of the text (for example, Python code, which is sensitive to indentation and spacing). ```

In [7]:
# we have splitted the words in raw_text nbut they are still a list of strings
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed[:10]

['I', ' ', 'HAD', ' ', 'always', ' ', 'thought', ' ', 'Jack', ' ']

In [8]:
# Here we remove whitespaces from the list of strings(words)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


## 2.3 Converting tokens into token IDs

### Let's create a random list of words and play with it before going for tokenization

In [9]:
random_list_of_words = ['Hey', 'How', 'are', 'you', 'How', 'are' ,'your', 'parents']
random_list_of_words

['Hey', 'How', 'are', 'you', 'How', 'are', 'your', 'parents']

In [10]:
type(random_list_of_words)

list

In [11]:
# now there is not need to carry the same word in our list while focusing on tokenization, why because on tokenwork would have a unique token id
# so we need a set of token/words not list - so that we get a list of unique words - utlizing property of data type set
unique_random_list_of_words = set(random_list_of_words)

In [12]:
all_words = unique_random_list_of_words

In [13]:
type(unique_random_list_of_words)

set

In [14]:
vocab_size = len(all_words)
vocab_size

6

In [15]:
# let's do the same for preprocessed
set_of_words = set(preprocessed)
sorted_set_of_words = sorted(set_of_words)

In [16]:
sorted_set_of_words[:30]

['!',
 '"',
 "'",
 '(',
 ')',
 ',',
 '--',
 '.',
 ':',
 ';',
 '?',
 'A',
 'Ah',
 'Among',
 'And',
 'Are',
 'Arrt',
 'As',
 'At',
 'Be',
 'Begin',
 'Burlington',
 'But',
 'By',
 'Carlo',
 'Chicago',
 'Claude',
 'Come',
 'Croft',
 'Destroyed']

In [17]:
vocab_size = len(sorted_set_of_words)

### Listing 2.2 Creating a vocabulary

In [18]:
vocab = { token:integer for integer, token in enumerate(sorted_set_of_words)}
type(vocab)

dict

In [19]:
# see the elements in vocab dict
see_vocab_dict = {k: vocab[k] for k in list(vocab)[:30]}
see_vocab_dict

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '--': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'Ah': 12,
 'Among': 13,
 'And': 14,
 'Are': 15,
 'Arrt': 16,
 'As': 17,
 'At': 18,
 'Be': 19,
 'Begin': 20,
 'Burlington': 21,
 'But': 22,
 'By': 23,
 'Carlo': 24,
 'Chicago': 25,
 'Claude': 26,
 'Come': 27,
 'Croft': 28,
 'Destroyed': 29}

In [20]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i>=30:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)


### Listing 2.3 Implementing a simple text tokenizer

In [21]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int = vocab # stores the vocab as a class attribute for access in the encode and decode methods
        self.int_to_str = { i:s for s,i in vocab.items()} # Create an inverse vocabulary that maps token IDs back to original text tokens

    def encode(self, text): # Processes input text into token IDs 
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)  # Split the sequence into tokens
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ] # remove the whitespaces and keep the word and characters
        ids = [ self.str_to_int[s] for s in preprocessed ] # map tokens to ids
        print(preprocessed[:3], ids[:3])
        return ids

    def decode(self, ids): # Converts token IDs back into text 
        text = " ".join([self.int_to_str[i] for i in ids]) # Takes the list of strings generated by the comprehension and concatenates them into one string placing a . between each element.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1',text) # re.sub(pattern, replacement, string): A Python regular expression function that searches a string for a pattern and replaces it.
        return text
        

In [22]:
# let's use the tokenizer that we created
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride.""" # We will use this sequence later  text_2 = """I am Swapnil, I love learning over the time my interest to learn gpu is increasing but at the same time I think I should also explore new chips like ceberas and groq"""
ids = tokenizer.encode(text)
ids

['"', 'It', "'"] [1, 56, 2]


[1,
 56,
 2,
 850,
 988,
 602,
 533,
 746,
 5,
 1126,
 596,
 5,
 1,
 67,
 7,
 38,
 851,
 1108,
 754,
 793,
 7]

### Great the above encoding works 

1. We first create a vocab list for the-verdit corpus
    a. there we first converted corpus(group of sequences) into words/token - splitted them -> created a set of it -> gave each token/word an numerical id 
2. We then created a class with two function(encode and decode) and two variable ( that store the mapping of token to ids and ids to tokens
3. We tried to convert words in a sequence 'text' into token leveraging the tokenizer that we created


In [23]:
print(tokenizer.decode(ids)) # calling the decode method of class tokenizer

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [24]:
### Now let's test it with a sequence that has words which may not be the same from the verdict corpus
# let's use the tokenizer that we created
tokenizer = SimpleTokenizerV1(vocab)
text_2 = """I am Swapnil, I love learning over the time my interest to learn gpu is increasing but at the same time I think I should also explore new chips like ceberas and groq"""
ids = tokenizer.encode(text_2)
ids

KeyError: 'Swapnil'

### We see an error 

```KeyError: 'Swapnil'```

### What does that means?
```It means as per python - "Bro, while I was try9ng to locate the token id i couldnot find the key "Swapnil" - seems like the token/word "Swapnil" is not present in vocab itself, leave alone the mapping ```

### 2.4 Adding special context tokens

In [25]:
# Let's go back to the vocab that we had created earlier 
sorted_set_of_words[-30:] # last 30 words

['why',
 'wide',
 'widow',
 'wife',
 'wild',
 'wincing',
 'window-curtains',
 'wish',
 'with',
 'without',
 'wits',
 'woman',
 'women',
 'won',
 'wonder',
 'wondered',
 'word',
 'work',
 'working',
 'worth',
 'would',
 'wouldn',
 'year',
 'years',
 'yellow',
 'yet',
 'you',
 'younger',
 'your',
 'yourself']

In [26]:
len(sorted_set_of_words)

1130

In [27]:
# now let's see if the vocab has word "Swapnil" or not
print("Swapnil" in sorted_set_of_words)
print("younger" in sorted_set_of_words)
print("wondered" in sorted_set_of_words)
# seems like it doesn't have

False
True
True


In [28]:
type(sorted_set_of_words)

list

In [29]:
#Here we are just adding these special token and giving them ID - as of now we haven't coded for logic how unknown token would be mapped to unknown tokens or special tokens
all_tokens = sorted_set_of_words
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}


In [30]:
see_vocab_dict = {k: vocab[k] for k in list(vocab)[-30:]}
see_vocab_dict

{'widow': 1102,
 'wife': 1103,
 'wild': 1104,
 'wincing': 1105,
 'window-curtains': 1106,
 'wish': 1107,
 'with': 1108,
 'without': 1109,
 'wits': 1110,
 'woman': 1111,
 'women': 1112,
 'won': 1113,
 'wonder': 1114,
 'wondered': 1115,
 'word': 1116,
 'work': 1117,
 'working': 1118,
 'worth': 1119,
 'would': 1120,
 'wouldn': 1121,
 'year': 1122,
 'years': 1123,
 'yellow': 1124,
 'yet': 1125,
 'you': 1126,
 'younger': 1127,
 'your': 1128,
 'yourself': 1129,
 '<|endoftext|>': 1130,
 '<|unk|>': 1131}

### Listing 2.4 A simple text tokenizer that handles unknown words

In [31]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab # a variable that have the vocab
        self.int_to_str = { i:s for s,i in vocab.items()} # a variable that have 

    def encode(self, text): # method to encode the new text as per vocab word/token to token ID mapping
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) # Here goes the sequence for splitting
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ] # removing spaces from list of words(including spaces) that was stored in preprocessed list
        preprocessed = [ item if item in self.str_to_int else "<|unk|>" for item in preprocessed ] # mapping tokens to token id and for unknown token(not present in vocab list mapping to token id of "<|unk|>" token id
        ids = [ self.str_to_int[s] for s in preprocessed ] 
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids]) # mapping token ids back to tokens/words and joining them with space in between
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text) # remove the space between word followed by puntuation mark
        return text


In [32]:
### Now let's test it with a sequence that has words which may not be the same from the verdict corpus
# let's use the tokenizer that we created
tokenizer = SimpleTokenizerV2(vocab)
text_2 = """I am Swapnil, I love learning over the time my interest to learn gpu is increasing but at the same time I think I should also explore new chips like ceberas and groq"""
ids = tokenizer.encode(text_2)
ids

[53,
 150,
 1131,
 5,
 53,
 1131,
 1131,
 741,
 988,
 1011,
 697,
 1131,
 1016,
 1131,
 1131,
 584,
 1131,
 239,
 180,
 988,
 852,
 1011,
 53,
 998,
 53,
 879,
 1131,
 1131,
 1131,
 1131,
 628,
 1131,
 157,
 1131]

In [33]:
print(tokenizer.decode(tokenizer.encode(text_2)))

I am <|unk|>, I <|unk|> <|unk|> over the time my <|unk|> to <|unk|> <|unk|> is <|unk|> but at the same time I think I should <|unk|> <|unk|> <|unk|> <|unk|> like <|unk|> and <|unk|>


In [34]:
text1 = "Hello, Bro do you like tea? "
text2 = " Bhai I love tea, masala tea specifically."
text = "<|endoftext|>".join((text1, text2))
print(text)

Hello, Bro do you like tea? <|endoftext|> Bhai I love tea, masala tea specifically.


In [35]:
print(tokenizer.encode(text))

[1131, 5, 1131, 355, 1126, 628, 975, 10, 1130, 1131, 53, 1131, 975, 5, 1131, 975, 1131, 7]


In [36]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, <|unk|> do you like tea? <|endoftext|> <|unk|> I <|unk|> tea, <|unk|> tea <|unk|>.


## 2.5 Byte pair encoding

In [37]:
!pip install tiktoken

In [38]:
from importlib.metadata import version
import tiktoken
print("tiktoken.version:", version("tiktoken"))

tiktoken.version: 0.12.0


In [39]:
tokenizer = tiktoken.get_encoding("gpt2")

In [40]:
text = ("Let’s look at a more sophisticated tokenization scheme based on a concept called byte pair encoding (BPE). The BPE tokenizer was used to train LLMs such as GPT-2, GPT-3, and the original model used in ChatGPT.")
integers = tokenizer.encode(text)

In [41]:
integers[:10]

[5756, 447, 247, 82, 804, 379, 257, 517, 13767, 11241]

In [42]:
text1 = "Hello, Bro do you like tea? "
text2 = " Bhai I love tea, masala tea specifically."
text = "<|endoftext|>".join((text1, text2))

integers = tokenizer.encode(text, allowed_special = { "<|endoftext|>", "<|unk|>"})
integers[:10]

[15496, 11, 2806, 466, 345, 588, 8887, 30, 220, 50256]

In [43]:
# let's detokenize
strings = tokenizer.decode(integers)
strings

'Hello, Bro do you like tea? <|endoftext|> Bhai I love tea, masala tea specifically.'

In [44]:
# Let's test the limit of this tokenizer("gpt2")
text1 = "Hello, Bro do you like tea? "
text2 = " Bhai I love tea, masala tea specifically, yoyo honey किंकर्तव्यविमूढ़ मर्मभेदी लोह-पथ-गामिनी kamina dhust abcdefghijklmnopqrstuvwxyz."
text = "<|endoftext|>".join((text1, text2))

integers = tokenizer.encode(text, allowed_special = { "<|endoftext|>", "<|unk|>"})
strings = tokenizer.decode(integers)
strings

'Hello, Bro do you like tea? <|endoftext|> Bhai I love tea, masala tea specifically, yoyo honey किंकर्तव्यविमूढ़ मर्मभेदी लोह-पथ-गामिनी kamina dhust abcdefghijklmnopqrstuvwxyz.'

### It works but how 
```The algorithm underlying BPE breaks down words that aren’t in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words. So, thanks to the BPE algorithm, if the tokenizer encounters an unfamiliar word during tokenization, it can represent it as a sequence of subword tokens or characters, ```

``` If you see below the tokens are much more than number of words, let's see what's happening```

In [45]:
integers

[15496,
 11,
 2806,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 16581,
 1872,
 314,
 1842,
 8887,
 11,
 12422,
 6081,
 8887,
 5734,
 11,
 331,
 726,
 78,
 12498,
 28225,
 243,
 11976,
 123,
 11976,
 224,
 11976,
 243,
 11976,
 108,
 24231,
 235,
 11976,
 97,
 11976,
 113,
 24231,
 235,
 11976,
 107,
 11976,
 113,
 11976,
 123,
 11976,
 106,
 24231,
 224,
 11976,
 95,
 11976,
 120,
 28225,
 106,
 11976,
 108,
 24231,
 235,
 11976,
 106,
 11976,
 255,
 24231,
 229,
 11976,
 99,
 24231,
 222,
 28225,
 110,
 24231,
 233,
 11976,
 117,
 12,
 11976,
 103,
 11976,
 98,
 12,
 11976,
 245,
 48077,
 11976,
 106,
 11976,
 123,
 11976,
 101,
 24231,
 222,
 479,
 18891,
 34590,
 436,
 450,
 66,
 4299,
 456,
 2926,
 41582,
 10295,
 404,
 80,
 81,
 301,
 14795,
 86,
 5431,
 89,
 13]

In [46]:
tokenizer.decode_bytes(integers)

b'Hello, Bro do you like tea? <|endoftext|> Bhai I love tea, masala tea specifically, yoyo honey \xe0\xa4\x95\xe0\xa4\xbf\xe0\xa4\x82\xe0\xa4\x95\xe0\xa4\xb0\xe0\xa5\x8d\xe0\xa4\xa4\xe0\xa4\xb5\xe0\xa5\x8d\xe0\xa4\xaf\xe0\xa4\xb5\xe0\xa4\xbf\xe0\xa4\xae\xe0\xa5\x82\xe0\xa4\xa2\xe0\xa4\xbc \xe0\xa4\xae\xe0\xa4\xb0\xe0\xa5\x8d\xe0\xa4\xae\xe0\xa4\xad\xe0\xa5\x87\xe0\xa4\xa6\xe0\xa5\x80 \xe0\xa4\xb2\xe0\xa5\x8b\xe0\xa4\xb9-\xe0\xa4\xaa\xe0\xa4\xa5-\xe0\xa4\x97\xe0\xa4\xbe\xe0\xa4\xae\xe0\xa4\xbf\xe0\xa4\xa8\xe0\xa5\x80 kamina dhust abcdefghijklmnopqrstuvwxyz.'

In [47]:
tokenizer.decode_tokens_bytes(integers)

[b'Hello',
 b',',
 b' Bro',
 b' do',
 b' you',
 b' like',
 b' tea',
 b'?',
 b' ',
 b'<|endoftext|>',
 b' Bh',
 b'ai',
 b' I',
 b' love',
 b' tea',
 b',',
 b' mas',
 b'ala',
 b' tea',
 b' specifically',
 b',',
 b' y',
 b'oy',
 b'o',
 b' honey',
 b' \xe0\xa4',
 b'\x95',
 b'\xe0\xa4',
 b'\xbf',
 b'\xe0\xa4',
 b'\x82',
 b'\xe0\xa4',
 b'\x95',
 b'\xe0\xa4',
 b'\xb0',
 b'\xe0\xa5',
 b'\x8d',
 b'\xe0\xa4',
 b'\xa4',
 b'\xe0\xa4',
 b'\xb5',
 b'\xe0\xa5',
 b'\x8d',
 b'\xe0\xa4',
 b'\xaf',
 b'\xe0\xa4',
 b'\xb5',
 b'\xe0\xa4',
 b'\xbf',
 b'\xe0\xa4',
 b'\xae',
 b'\xe0\xa5',
 b'\x82',
 b'\xe0\xa4',
 b'\xa2',
 b'\xe0\xa4',
 b'\xbc',
 b' \xe0\xa4',
 b'\xae',
 b'\xe0\xa4',
 b'\xb0',
 b'\xe0\xa5',
 b'\x8d',
 b'\xe0\xa4',
 b'\xae',
 b'\xe0\xa4',
 b'\xad',
 b'\xe0\xa5',
 b'\x87',
 b'\xe0\xa4',
 b'\xa6',
 b'\xe0\xa5',
 b'\x80',
 b' \xe0\xa4',
 b'\xb2',
 b'\xe0\xa5',
 b'\x8b',
 b'\xe0\xa4',
 b'\xb9',
 b'-',
 b'\xe0\xa4',
 b'\xaa',
 b'\xe0\xa4',
 b'\xa5',
 b'-',
 b'\xe0\xa4',
 b'\x97',
 b'\xe0\xa4\xbe',
 

## 2.6 Data sampling with a sliding window

In [48]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [49]:
enc_sample = enc_text[50:]
enc_sample

[290,
 4920,
 2241,
 287,
 257,
 4489,
 64,
 319,
 262,
 34686,
 41976,
 13,
 357,
 10915,
 314,
 2138,
 1807,
 340,
 561,
 423,
 587,
 10598,
 393,
 28537,
 2014,
 198,
 198,
 1,
 464,
 6001,
 286,
 465,
 13476,
 1,
 438,
 5562,
 373,
 644,
 262,
 1466,
 1444,
 340,
 13,
 314,
 460,
 3285,
 9074,
 13,
 46606,
 536,
 5469,
 438,
 14363,
 938,
 4842,
 1650,
 353,
 438,
 2934,
 489,
 3255,
 465,
 48422,
 540,
 450,
 67,
 3299,
 13,
 366,
 5189,
 1781,
 340,
 338,
 1016,
 284,
 3758,
 262,
 1988,
 286,
 616,
 4286,
 705,
 1014,
 510,
 26,
 475,
 314,
 836,
 470,
 892,
 286,
 326,
 11,
 1770,
 13,
 8759,
 2763,
 438,
 1169,
 2994,
 284,
 943,
 17034,
 318,
 477,
 314,
 892,
 286,
 526,
 383,
 1573,
 11,
 319,
 9074,
 13,
 536,
 5469,
 338,
 11914,
 11,
 33096,
 663,
 4808,
 3808,
 62,
 355,
 996,
 484,
 547,
 12548,
 287,
 281,
 13079,
 410,
 12523,
 286,
 22353,
 13,
 843,
 340,
 373,
 407,
 691,
 262,
 9074,
 13,
 536,
 48819,
 508,
 25722,
 276,
 13,
 11161,
 407,
 262,
 40123,
 18113,


In [50]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size]

In [51]:
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287]


In [52]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "--->", desired)

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


In [53]:
# This is interesting since desired has a dataype of int you need desired to be inside [] to make it work as tokenzier decode does not accept int valaue as input
print(type(context))
print(type(desired))

<class 'list'>
<class 'int'>


In [54]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode(desired))

TypeError: argument 'tokens': 'int' object cannot be converted to 'Sequence'

In [55]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


## Listing 2.5 A dataset for batched inputs and targets

In [61]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):

        self.input_ids = [] # A variable to store input_ids
        self. target_ids = [] # A variable to store target ids

        token_ids = tokenizer.encode(txt) # tokenizing the whole txt
        for i in range(0, len(token_ids) - max_length, stride): # a for loop that will start from token at 0 address till token address at token_legth - max_lenght, with next window at stride gap
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1: i+max_length+1]

            self.input_ids.append(torch.tensor(input_chunk)) # converted input_chunks(token ids) -> Tensors -> appended them to a list of input_ids
            #print(self.input_ids)
            self.target_ids.append(torch.tensor(target_chunk))
            #print(self.target_ids)


    def __len__(self):
        return len(self.input_ids) # return length of input_ids

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx] #return token at position idx

## Listing 2.6 A data loader to generate batches with input targets pairs

In [62]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride = 128, shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2") # Initiliaze tokenizer
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # get all the tokens in form of input_ids, target_ids
    dataloader = DataLoader(
                    dataset,
                    batch_size = batch_size,
                    shuffle = shuffle,
                    drop_last = drop_last, # drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training. 
                    num_workers = num_workers # The number of CPU processes to use for preprocessing 
                )

    return dataloader

In [63]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1( raw_text, batch_size=1, max_length = 4, stride = 1, shuffle = False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

In [64]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [65]:
first_batch

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

### Exercise: 2.2. Data Loaders with Different Strides 

In [69]:
dataloader = create_dataloader_v1(raw_text, batch_size = 8, max_length = 4, stride = 4, shuffle = False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs:\n", inputs)
print("\nTargets:\n", targets)


Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 Create Token EMbeddings

In [72]:
input_ids = torch.tensor([2, 3, 5, 1])
input_ids

tensor([2, 3, 5, 1])

In [75]:
vocab_size = 6
output_dim = 3

In [77]:
torch.manual_seed(35)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
embedding_layer

Embedding(6, 3)

In [82]:
type(embedding_layer)

torch.nn.modules.sparse.Embedding

In [78]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.1838,  0.1832, -0.1515],
        [-1.4119, -0.2994, -2.1561],
        [ 0.4940,  2.4608, -1.2028],
        [ 0.8062, -1.8353,  0.5851],
        [-2.5162,  0.1458,  1.5299],
        [ 0.9691, -0.7181, -1.6961]], requires_grad=True)


In [81]:
print(embedding_layer(torch.tensor([3])))

tensor([[ 0.8062, -1.8353,  0.5851]], grad_fn=<EmbeddingBackward0>)


In [83]:
print(embedding_layer(input_ids))

tensor([[ 0.4940,  2.4608, -1.2028],
        [ 0.8062, -1.8353,  0.5851],
        [ 0.9691, -0.7181, -1.6961],
        [-1.4119, -0.2994, -2.1561]], grad_fn=<EmbeddingBackward0>)


In [84]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [89]:
token_embedding_layer.weight

Parameter containing:
tensor([[ 0.4738, -1.2670,  0.1890,  ..., -0.0151,  1.0619,  0.9503],
        [-0.5536, -0.3598,  2.0448,  ...,  1.1213, -0.8541,  1.5768],
        [-0.2958,  1.1485, -0.5220,  ..., -1.4602,  0.7203,  1.8309],
        ...,
        [-0.7133, -0.6827,  0.2921,  ..., -0.9157,  0.1481, -0.3095],
        [ 0.9396, -0.1979, -0.4691,  ...,  0.8093,  0.2358, -0.1575],
        [-0.6065,  0.6228,  0.6514,  ...,  0.9753, -0.6663,  0.3550]],
       requires_grad=True)

In [91]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size = 8, max_length = max_length, stride = max_length, shuffle = False)

data_iter = iter(dataloader)
inputs, targets  = next(data_iter)

print("Token IDs:\n", inputs)
print("\nInput shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Input shape:
 torch.Size([8, 4])


In [95]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings)
print(token_embeddings.shape)

tensor([[[-0.2225, -2.3800,  0.5469,  ...,  0.0250,  2.0764, -1.7234],
         [-0.4299,  1.0888,  0.1722,  ...,  0.0118,  0.5073, -0.6582],
         [ 2.4525, -1.3625, -0.8522,  ...,  0.2970,  0.8568,  0.3405],
         [ 1.6229,  0.7399, -0.4446,  ..., -1.7530, -2.0467, -0.4346]],

        [[ 0.0983, -0.8326, -0.3530,  ..., -1.4617,  0.3544, -0.8011],
         [-0.2659, -1.1000, -1.2763,  ...,  0.4693, -2.1453, -0.0921],
         [ 0.8061, -1.8816,  0.4036,  ..., -0.6065, -1.6600, -0.3166],
         [ 1.4263,  0.9142, -1.3780,  ...,  0.2679,  0.9763,  1.0848]],

        [[ 0.0344,  0.2606,  1.5168,  ..., -1.6064,  0.6736, -0.4660],
         [-1.4036,  0.9354,  0.9391,  ...,  2.0811, -1.1356,  0.5407],
         [ 0.6219, -0.2617,  1.8057,  ..., -0.8084,  0.7810, -0.5205],
         [ 1.3912, -0.6333, -1.6214,  ..., -0.3263, -0.2013, -0.2371]],

        ...,

        [[ 1.5298, -0.3179,  0.0870,  ..., -0.0738,  0.4288, -0.1924],
         [ 0.1059,  0.2877, -0.7642,  ..., -0.6263,  0.24

In [94]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings)
print(pos_embeddings.shape)

tensor([[-2.0042, -0.7169, -0.3546,  ..., -0.7060, -1.9739, -0.4478],
        [-1.8861, -1.4693, -1.1563,  ..., -1.1842, -0.4351, -0.1382],
        [-1.2703,  0.9082,  1.5550,  ...,  0.9843,  0.0096,  0.9233],
        [-0.7661,  0.0700,  1.7954,  ...,  0.7077, -0.5644,  0.0302]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([4, 256])


In [97]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(input_embeddings)

torch.Size([8, 4, 256])
tensor([[[-2.2267, -3.0969,  0.1923,  ..., -0.6810,  0.1025, -2.1712],
         [-2.3160, -0.3805, -0.9841,  ..., -1.1725,  0.0722, -0.7964],
         [ 1.1822, -0.4543,  0.7028,  ...,  1.2813,  0.8664,  1.2639],
         [ 0.8568,  0.8100,  1.3508,  ..., -1.0452, -2.6111, -0.4044]],

        [[-1.9059, -1.5495, -0.7076,  ..., -2.1678, -1.6196, -1.2489],
         [-2.1521, -2.5693, -2.4325,  ..., -0.7149, -2.5804, -0.2303],
         [-0.4642, -0.9733,  1.9586,  ...,  0.3778, -1.6504,  0.6067],
         [ 0.6603,  0.9842,  0.4174,  ...,  0.9756,  0.4118,  1.1151]],

        [[-1.9698, -0.4563,  1.1622,  ..., -2.3124, -1.3004, -0.9138],
         [-3.2898, -0.5340, -0.2172,  ...,  0.8969, -1.5707,  0.4025],
         [-0.6484,  0.6465,  3.3607,  ...,  0.1759,  0.7906,  0.4028],
         [ 0.6251, -0.5633,  0.1740,  ...,  0.3814, -0.7657, -0.2068]],

        ...,

        [[-0.4745, -1.0348, -0.2676,  ..., -0.7798, -1.5451, -0.6402],
         [-1.7803, -1.1816, -1.92